In [6]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize

# 1. Setup tickers and data
tickers = ["SPY", "TLT", "GLD", "EFA"]
prices = yf.download(tickers, start="2018-01-01")["Close"]
returns = prices.pct_change().dropna()

# 2. Calculate annual returns and covariance matrix
annual_return = returns.mean() * 252
annual_cov = returns.cov() * 252

# 3. Portfolio functions
def portfolio_stats(weights):
    expected_return = np.dot(weights, annual_return)
    volatility = np.sqrt(np.dot(weights.T, np.dot(annual_cov, weights)))
    return expected_return, volatility

def objective(weights):
    r, v = portfolio_stats(weights)
    return -(r / v)  # Negative Sharpe Ratio for minimization

# 4. Define constraints and bounds
# Equality constraint: Sum of weights must equal 1
constraints = {"type": "eq", "fun": lambda x: np.sum(x) - 1}

# Bound constraint: Each ticker must be between 0.05 (5%) and 1 (100%)
bounds = [(0.05, 1.0) for _ in range(len(tickers))]

# 5. Run optimization
# Starting guess of equal weights
initial_guess = np.repeat(1/len(tickers), len(tickers))

result = minimize(
    objective, 
    initial_guess, 
    bounds=bounds, 
    constraints=constraints
)

# 6. Display results
allocation = pd.DataFrame({"ETF": tickers, "Weight": result.x})
allocation = allocation.round(3)
print(allocation)

[*********************100%***********************]  4 of 4 completed

   ETF  Weight
0  SPY   0.050
1  TLT   0.504
2  GLD   0.396
3  EFA   0.050
